# Batch AI Defect Classification (T4 GPU)
Uses T4 GPU with Qwen2.5-3B-Instruct to classify 14k+ negative reviews.
Outputs `classified_reviews.csv`.

## Cell 1: Install Dependencies

In [ ]:
!pip install transformers accelerate pandas tqdm -q
print('Dependencies installed!')

## Cell 2: Upload CSV
If on Colab, an upload prompt will appear. If on Kaggle, upload `reviews_to_classify.csv` to the right-hand sidebar's working directory.

In [ ]:
import pandas as pd
import glob

# Automatically search everywhere in Kaggle for the file
possible_files = glob.glob('/kaggle/**/*.csv', recursive=True)
found = [f for f in possible_files if 'reviews_to_classify' in f.lower()]

if found:
    file_name = found[0]
    df = pd.read_csv(file_name)
    print(f'Loaded {len(df)} reviews from {file_name}')
else:
    print('ERROR: File not found! Please check the right-hand sidebar to ensure the file uploaded.')

## Cell 3: Load Model onto GPU

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {model_id}...')

tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto'
)
print(f'Model loaded on {model.device}!')

PROMPT = """You are an expert supply chain quality analyst evaluating a customer review (in Portuguese).
Your task is to analyze the text, translate it internally, and classify it into EXACTLY ONE of the following English categories:
1. Late Delivery
2. Damaged Goods
3. Defective Product
4. Wrong Item
5. Missing Parts
6. Poor Quality
7. Not Received
8. Rude Courier
9. Color / Size Mismatch
10. Poor Packaging
11. Communication Issues
12. Fraud / Counterfeit
13. Subjective Complaint (e.g. 'terrible taste', 'did not like', opinions)
14. Customer Error (e.g. bought wrong item)
15. No Defect (e.g. positive review or just asking a question)

Do NOT invent new categories. You MUST pick the best match from the list above.

Also assign a Severity Weight from 0.0 to 5.0. Use fine-grained continuous decimals (e.g., 2.3, 3.8, 4.5) to be precise.
CRITICAL RULE: If the category is 'Subjective Complaint', 'Customer Error', or 'No Defect', the Severity MUST be 1.0 or lower (0.0 - 1.0), regardless of how angry the customer is.
General Scale for other defects:
- 1.1-2.0: Minor inconvenience (e.g., slight box dent, late by 1 day).
- 2.1-3.5: Moderate issue (e.g., late by 4 days, wrong color but usable, low quality).
- 3.6-4.9: Major issue (e.g., shattered product, completely wrong item, missing main part).
- 5.0: Critical failure / scam (e.g., never arrived, empty box, blatant fraud).

Respond EXACTLY in this format with no other words: Category|Severity

Examples:
Review: 'O gosto é horrível, odiei.'
Output: Subjective Complaint|0.5
Review: 'A tela do celular chegou toda trincada e não liga.'
Output: Damaged Goods|4.8
Review: 'Atrasou 2 dias mas o produto é bom.'
Output: Late Delivery|2.2

Review: '{text}'"""


In [ ]:
PROMPT = """You are a supply chain quality analyst evaluating a customer review (in Portuguese).
Identify the core defect category. You can choose from:
- Late Delivery
- Damaged Goods
- Wrong Item
- Poor Quality

If none of those accurately fit, INVENT a new concise category (1-3 words max, in English) like 'Rude Courier' or 'Missing Parts'.

        "Also assign a highly precise and fluid Severity Weight from 1.0 to 5.0 based on how bad the experience is.\n",
        "Use decimals (e.g., 3.7, 4.2, 2.1) to capture the exact nuance of the severity, do NOT just stick to whole numbers. Here is a general guide:\n",
- 1.0: Minor inconvenience (e.g. slight package scuff, minor customer question, minor product variance).
- 2.0: Minor issue / mild delay (e.g. late by 1 day, small cosmetic defect).
- 3.0: Moderate defect or notable delay (e.g. late by 2-5 days, wrong size, low quality but functional).
- 4.0: Major issue (e.g. extremely late delivery, broken/damaged product, completely wrong product sent).
- 5.0: Critical failure / scam (e.g. item never arrived, package was empty, shattered/totally unusable item).

Respond EXACTLY in this format with no other words: Category|Severity
Example 1: Damaged Goods|4.0
Example 2: Rude Courier|3.5

Review: '{text}'"""

sample_texts = df['ReviewText'].fillna('').astype(str).tolist()[:5]
sample_prompts = [
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': PROMPT.format(text=t)}],
        tokenize=False, add_generation_prompt=True
    ) for t in sample_texts
]

inputs = tokenizer(sample_prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=15, do_sample=False)
input_len = inputs.input_ids.shape[1]
results = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

print('--- Sanity Check Results ---')
for i, (review, result) in enumerate(zip(sample_texts, results)):
    print(f'{i+1}. Review: {review[:80]}...')
    print(f'   Output: {repr(result.strip())}')
    print()

## Cell 5: Full Batch Inference
If sanity check looks correct, run this cell to process all reviews.

In [ ]:
from tqdm.notebook import tqdm
import csv

texts = df['ReviewText'].fillna('').astype(str).tolist()
inspection_ids = df['InspectionID'].tolist()

print('Formatting prompts...')
prompts = []
for t in tqdm(texts, desc='Formatting'):
    prompts.append(
        tokenizer.apply_chat_template(
            [{'role': 'user', 'content': PROMPT.format(text=t)}],
            tokenize=False, add_generation_prompt=True
        )
    )
print(f'All {len(prompts)} prompts ready.')

batch_size = 16
total_batches = (len(prompts) + batch_size - 1) // batch_size

with open('classified_reviews.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['InspectionID', 'DefectCategory', 'SeverityWeight'])

    for i in tqdm(range(0, len(prompts), batch_size), total=total_batches, desc='Classifying'):
        batch_prompts = prompts[i:i+batch_size]
        batch_ids = inspection_ids[i:i+batch_size]

        inputs = tokenizer(
            batch_prompts, return_tensors='pt',
            padding=True, truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=15, do_sample=False)

        input_len = inputs.input_ids.shape[1]
        generated_texts = tokenizer.batch_decode(
            outputs[:, input_len:], skip_special_tokens=True
        )

        for j, gen_text in enumerate(generated_texts):
            gen_text = gen_text.strip()
            if '|' in gen_text:
                cat, sev = gen_text.split('|', 1)
                cat = cat.strip()
                try:
                    sev = float(sev.strip())
                except:
                    sev = 2.5
            else:
                cat = gen_text[:30]
                sev = 2.5

            writer.writerow([batch_ids[j], cat, sev])

        f.flush()

print('\nClassification complete!')
result_df = pd.read_csv('classified_reviews.csv')
print(f'Total classified: {len(result_df)}')
print(result_df['DefectCategory'].value_counts().head(10))

## Cell 6: Download Result

In [ ]:
import os
output_file = 'classified_reviews.csv'
print(f'File saved to {os.path.abspath(output_file)}')
try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print('If you are on Kaggle, find the file in the right-hand sidebar under Output -> /kaggle/working/ and click the three dots to download it.')